In [2]:
pip install pyscipopt

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pyomo.environ import *

# ==========================================================
# MODEL
# ==========================================================

m = ConcreteModel(name="Crude_Scheduling_MINLP")


# ==========================================================
# SETS
# ==========================================================

m.FEEDS = Set(
    initialize=[
        "F1",
        "F2",
        "F3",
        "F4",
        "F5",
        "F6",
    ]
)

m.TANKS = Set(
    initialize=[
        "T1",
        "T2",
        "T3",
    ]
)


# ==========================================================
# DATA
# ==========================================================

feed_max_flow = {
    "F1": 60,
    "F2": 70,
    "F3": 50,
    "F4": 80,
    "F5": 60,
    "F6": 40,
}

feed_sulfur = {
    "F1": 0.20,
    "F2": 0.35,
    "F3": 0.60,
    "F4": 1.10,
    "F5": 1.50,
    "F6": 2.20,
}

feed_margin = {
    "F1": 14,
    "F2": 12,
    "F3": 18,
    "F4": 25,
    "F5": 30,
    "F6": 35,
}

feed_startup_cost = {
    "F1": 100,
    "F2": 100,
    "F3": 120,
    "F4": 150,
    "F5": 180,
    "F6": 200,
}

MIN_FEED_NOMINATION = 10

CDU_MIN_THROUGHPUT = 220
CDU_MAX_THROUGHPUT = 250

CDU_SULFUR_SPEC = 0.85


# ==========================================================
# VARIABLES
# ==========================================================

# feed -> tank flow

m.feed_to_tank_flow = Var(
    m.FEEDS,
    m.TANKS,
    domain=NonNegativeReals
)

# tank -> CDU flow

m.tank_to_cdu_flow = Var(
    m.TANKS,
    domain=NonNegativeReals
)

# sulfur concentration in each tank

m.tank_sulfur = Var(
    m.TANKS,
    bounds=(0, 3)
)

# blended sulfur entering CDU

m.cdu_sulfur = Var(
    bounds=(0, 3)
)

# binary feed activation

m.feed_active = Var(
    m.FEEDS,
    domain=Binary
)


# ==========================================================
# EXPRESSIONS
# ==========================================================

def cdu_flow_rule(m):
    return sum(
        m.tank_to_cdu_flow[tank]
        for tank in m.TANKS
    )

m.cdu_flow = Expression(
    rule=cdu_flow_rule
)


# ==========================================================
# FEED ACTIVATION CONSTRAINTS
# ==========================================================

def feed_capacity_rule(m, feed):

    return (

        sum(
            m.feed_to_tank_flow[feed, tank]
            for tank in m.TANKS
        )

        <=

        feed_max_flow[feed]
        *
        m.feed_active[feed]

    )


m.feed_capacity = Constraint(
    m.FEEDS,
    rule=feed_capacity_rule
)


def feed_minimum_nomination_rule(m, feed):

    return (

        sum(
            m.feed_to_tank_flow[feed, tank]
            for tank in m.TANKS
        )

        >=

        MIN_FEED_NOMINATION
        *
        m.feed_active[feed]

    )


m.feed_minimum_nomination = Constraint(
    m.FEEDS,
    rule=feed_minimum_nomination_rule
)


# ==========================================================
# TANK MATERIAL BALANCES
# ==========================================================

def tank_material_balance_rule(m, tank):

    return (

        sum(
            m.feed_to_tank_flow[feed, tank]
            for feed in m.FEEDS
        )

        ==

        m.tank_to_cdu_flow[tank]

    )


m.tank_material_balance = Constraint(
    m.TANKS,
    rule=tank_material_balance_rule
)


# ==========================================================
# TANK SULFUR BALANCES
# ==========================================================
#
# pooling constraint
#
# sum(feed_sulfur * flow)
# =
# tank_sulfur * tank_flow
#
# ==========================================================

def tank_sulfur_balance_rule(m, tank):

    incoming_sulfur = sum(

        feed_sulfur[feed]
        *
        m.feed_to_tank_flow[feed, tank]

        for feed in m.FEEDS

    )

    outgoing_sulfur = (

        m.tank_sulfur[tank]
        *
        m.tank_to_cdu_flow[tank]

    )

    return incoming_sulfur == outgoing_sulfur


m.tank_sulfur_balance = Constraint(
    m.TANKS,
    rule=tank_sulfur_balance_rule
)


# ==========================================================
# CDU SULFUR BALANCE
# ==========================================================
#
# sum(tank sulfur contribution)
# =
# cdu sulfur * total cdu flow
#
# ==========================================================

def cdu_sulfur_balance_rule(m):

    incoming_sulfur = sum(

        m.tank_sulfur[tank]
        *
        m.tank_to_cdu_flow[tank]

        for tank in m.TANKS

    )

    outgoing_sulfur = (

        m.cdu_sulfur
        *
        m.cdu_flow

    )

    return incoming_sulfur == outgoing_sulfur


m.cdu_sulfur_balance = Constraint(
    rule=cdu_sulfur_balance_rule
)


# ==========================================================
# CDU THROUGHPUT CONSTRAINTS
# ==========================================================

m.cdu_minimum_throughput = Constraint(
    expr=
    m.cdu_flow >= CDU_MIN_THROUGHPUT
)

m.cdu_maximum_throughput = Constraint(
    expr=
    m.cdu_flow <= CDU_MAX_THROUGHPUT
)


# ==========================================================
# CDU SULFUR SPECIFICATION
# ==========================================================

m.cdu_sulfur_specification = Constraint(
    expr=
    m.cdu_sulfur <= CDU_SULFUR_SPEC
)


# ==========================================================
# OBJECTIVE
# ==========================================================

def profit_rule(m):

    feed_profit = sum(

        feed_margin[feed]
        *
        m.feed_to_tank_flow[feed, tank]

        for feed in m.FEEDS
        for tank in m.TANKS

    )

    feed_activation_penalty = sum(

        feed_startup_cost[feed]
        *
        m.feed_active[feed]

        for feed in m.FEEDS

    )

    return (

        feed_profit
        -
        feed_activation_penalty

    )


m.total_profit = Objective(
    rule=profit_rule,
    sense=maximize
)


# ==========================================================
# DISPLAY MODEL SIZE
# ==========================================================

print("Model created.")
print("Feeds :", len(m.FEEDS))
print("Tanks :", len(m.TANKS))
print("Variables :", m.nvariables())
print("Constraints :", m.nconstraints())

Model created.
Feeds : 6
Tanks : 3
Variables : 31
Constraints : 22


In [4]:
from pyomo.opt import SolverFactory, TerminationCondition, SolverStatus

solver = SolverFactory("scip")

results = solver.solve(
    m,
    tee=True
)

print("\n====================")
print("Solver Status")
print("====================")
print(results.solver.status)

print("\n====================")
print("Termination")
print("====================")
print(results.solver.termination_condition)

SCIP version 9.2.2 [precision: 8 byte] [memory: block] [mode: optimized] [LP solver: SoPlex 7.1.4] [GitHash: 416226a4f8]
Copyright (c) 2002-2025 Zuse Institute Berlin (ZIB)

External libraries: 
  SoPlex 7.1.4         Linear programming solver developed at Zuse Institute Berlin (soplex.zib.de) [GitHash: 7c53d552]
  CppAD 20180000.0     Algorithmic Differentiation of C++ algorithms developed by B. Bell (github.com/coin-or/CppAD)
  ZLIB 1.3.1           General purpose compression library by J. Gailly and M. Adler (zlib.net)
  GMP 6.3.0            GNU Multiple Precision Arithmetic Library developed by T. Granlund (gmplib.org)
  ZIMPL 3.6.2          Zuse Institute Mathematical Programming Language developed by T. Koch (zimpl.zib.de)
  AMPL/MP 690e9e7      AMPL .nl file reader library (github.com/ampl/mp)
  PaPILO 2.4.2         parallel presolve for integer and linear optimization (github.com/scipopt/papilo) (built with TBB) [GitHash: 4b399c4c]
  Nauty 2.8.8          Computing Graph Automor

In [1]:
from pyomo.environ import *

print(SolverFactory("scip").available())

True


In [ ]:
# conda install -c conda-forge scip

3 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\TONGX\AppData\Local\miniconda3\envs\idaes-env

  added / updated specs:
    - scip


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.7.22  |       h4c7d964_0         129 KB  conda-forge
    cppad-20250000.2           |       he0c23c2_0         584 KB  conda-forge
    gmp-6.3.0                  |       hfeafd45_2         554 KB  conda-forge
    ipopt-3.14.17              |       h812a801_2         918 KB  conda-forge
    libboost-1.86.0            |       h444863b_2         2.3 MB  conda-forge
    mpfr-4.2.2                 |       h883a981_0         717 KB  conda-forge
    mumps-seq-5.7.3            |      hbaa6519_10         7.5 MB  conda-forge
    openssl-3.6.3             



==> WARNING: A newer version of conda exists. <==
    current version: 25.7.0
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda


